# HyPhiX – complete experiments on Google Colab
**Before you start**
1. Upload `HyPhiX-Twitter-Sentiment.zip` to the top level of your Google Drive (*My Drive*).
2. Runtime → Change runtime type → **T4 GPU** (or better).
3. Run the cells from top to bottom.

All results are written **directly to Google Drive** (`My Drive/HyPhiX_runs` and `My Drive/HyPhiX_reports`), so nothing is lost if Colab disconnects.
Every training cell **skips runs that are already finished**, so after a disconnection simply reconnect and run all cells again.

## 1. Connect Google Drive and unpack the repository

In [ ]:
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
ZIP = f'{DRIVE}/HyPhiX-Twitter-Sentiment.zip'      # change this if the zip is in a sub-folder
REPO = '/content/HyPhiX-Twitter-Sentiment'
assert os.path.exists(ZIP), f'Zip file not found: {ZIP} - upload it to My Drive first'
if not os.path.exists(REPO):
    zipfile.ZipFile(ZIP).extractall('/content')
%cd {REPO}

## 2. Install the libraries and check the GPU

In [ ]:
!pip -q install -r requirements.txt
import nltk
for r in ['stopwords','wordnet','omw-1.4','averaged_perceptron_tagger','averaged_perceptron_tagger_eng','opinion_lexicon']:
    nltk.download(r, quiet=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Save all results to Google Drive
`runs/` and `reports/` are linked to folders in your Drive, so every finished run is kept permanently.

In [ ]:
import os, shutil
for d in ['runs', 'reports']:
    target = f'{DRIVE}/HyPhiX_{d}'
    os.makedirs(target, exist_ok=True)
    if os.path.islink(d):
        continue
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.symlink(target, d)
print('runs    ->', os.path.realpath('runs'))
print('reports ->', os.path.realpath('reports'))

### Helper: run a command only if its result does not exist yet

In [ ]:
import subprocess, sys
def run(cmd, done_file=None):
    """Run a script unless done_file already exists (resumes after a disconnection)."""
    if done_file and os.path.exists(done_file):
        print('already finished, skipping:', cmd)
        return
    print('>>>', cmd, flush=True)
    p = subprocess.run(cmd, shell=True)
    if p.returncode != 0:
        raise RuntimeError(f'command failed: {cmd}')

def train(model, extra='', name=None):
    name = name or f'{model}_seed2018'
    run(f'python scripts/train.py --model {model} {extra}', f'runs/{name}/metrics_validation_dedup.json')

## 4. Dataset statistics and leakage report

In [ ]:
run('python scripts/data_report.py', 'reports/data_report.json')
print(open('reports/data_report.json').read())

## 5. Baselines and hybrid models (Table 6, Figures 4–5)

In [ ]:
for m in ['cnn', 'bilstm', 'cb', 'bb']:
    train(m)
train('bert')          # about 1.5-2 hours on a T4

## 6. Hybrid models with SMOTE (Tables 7–9, Figure 7)

In [ ]:
train('cb', '--balance smote', 'cb_smote_seed2018')
train('bb', '--balance smote', 'bb_smote_seed2018')

In [ ]:
from IPython.display import Image, display
import json
for r in ['cb_seed2018', 'cb_smote_seed2018', 'bb_seed2018', 'bb_smote_seed2018']:
    m = json.load(open(f'runs/{r}/metrics_validation.json'))
    print(f"{r:20s} accuracy {m['accuracy']:.4f} CI {m['accuracy_ci_wilson'][0]:.4f}-{m['accuracy_ci_wilson'][1]:.4f} | macro-F1 {m['macro_f1']:.4f}")
display(Image('runs/cb_smote_seed2018/confusion_validation.png', width=420))

## 7. SHAP explanations (Figures 8–9)

In [ ]:
for r in ['cb_smote_seed2018', 'bb_smote_seed2018']:
    run(f'python scripts/explain_shap.py --run runs/{r} --eval-set validation',
        f'runs/{r}/shap_validation/shap_global_ranking.csv')
display(Image('runs/cb_smote_seed2018/shap_validation/shap_global_top_words.png', width=520))

## 8. Comparison tables, McNemar tests and figures

In [ ]:
RUNS = 'runs/cnn_seed2018 runs/bilstm_seed2018 runs/bert_seed2018 runs/cb_seed2018 runs/cb_smote_seed2018 runs/bb_seed2018 runs/bb_smote_seed2018'
LABELS = 'CNN BiLSTM BERT CB CB+SMOTE BB BB+SMOTE'
for s in ['validation', 'test', 'validation_dedup']:
    run(f'python scripts/compare_runs.py --eval-set {s} --runs {RUNS} --labels {LABELS}')
display(Image('reports/comparison/validation/accuracy.png', width=520))

## 9. Complete HyPhiX model and ablations (hypothesis H2)

In [ ]:
train('hyphix', '--fusion gate',   'hyphix_gate_seed2018')
train('hyphix', '--fusion concat', 'hyphix_concat_seed2018')
train('hyphix', '--fusion equal',  'hyphix_equal_seed2018')
train('hyphix', '--fusion gate --no-psycho', 'hyphix_gate_nopsy_seed2018')

## 10. Multi-seed experiments (mean ± SD, 95 % CI, paired tests)
About 3–4 hours per model on a T4.

In [ ]:
run('python scripts/multiseed.py --model cb --balance smote --seeds 2018 42 7 123 2024', 'reports/multiseed/cb_smote.json')
run('python scripts/multiseed.py --model bb --balance smote --seeds 2018 42 7 123 2024 --compare-to reports/multiseed/cb_smote.json',
    'reports/multiseed/bb_smote.json')

## 11. Leakage-aware protocol (grouped split)

In [ ]:
train('cb', '--balance smote --split group', 'cb_smote_group_seed2018')
train('bb', '--balance smote --split group', 'bb_smote_group_seed2018')

## 12. Package the results
Creates `My Drive/HyPhiX_results.zip` (model weights excluded). Send this file for the manuscript update.

In [ ]:
!cd /content && zip -qr {DRIVE}/HyPhiX_results.zip HyPhiX-Twitter-Sentiment/runs/ HyPhiX-Twitter-Sentiment/reports/ -x '*.pt'
print('saved:', f'{DRIVE}/HyPhiX_results.zip')